# Output integrity and physical consistency (regional run): `tas`, `tasmax`, `tasmin`, `pr`, `rsds`

Same three QA/QC parts as the production
[`output-integrity-checks.ipynb`](output-integrity-checks.ipynb) notebook -- output integrity
([#450](https://github.com/carbonplan/srm-downscaling/issues/450)), temperature ordering
`tasmin ≤ tas ≤ tasmax` ([#448](https://github.com/carbonplan/srm-downscaling/issues/448)), and
precipitation physical constraints
([#459](https://github.com/carbonplan/srm-downscaling/issues/459)) -- run instead against the
*regional (wide)* run: a small spatial subset (lat -31 to -26, lon 23 to 30, interior South Africa)
that carries every GCM (`CESM2-WACCM`, `MIROC-ES2H`, `UKESM`), scenario, variable, and ensemble
member, each in its own icechunk store on branch `full-regional-run`. This is a temporary check
against a scratch-bucket run, not a production artifact.

In [ ]:
import functools
import os

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
import coiled
import dask
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
import xarray as xr
import zarr
from dask_array.xarray import register

from srm.config import ClusterConfig
from srm.qaqc import VAR_SPATIAL_RANGES
from srm.validation import _open_output_datatree, resolve_member_time_bounds

os.environ["FRISKY_SUMMARY"] = "off"
from frisky import hijack

# Activate frisky's query-optimized dask array backend for xarray before opening any store,
# so every array uses one backend (see https://matthewrocklin.com/frisky-xarray/).
register()

zarr.config.set({"async.concurrency": 128})

# --- Run parameters: one icechunk store per GCM, same branch, small spatial subset ---
BUCKET = "carbonplan-scratch"
BRANCH = "full-regional-run"
PREFIXES = {
    "CESM2-WACCM": "srm/output/qa/CESM2-WACCM-ERA5-lat-31.0to-26.0_lon23.0to30.0.icechunk",
    "MIROC-ES2H": "srm/output/qa/MIROC-ES2H-ERA5-lat-31.0to-26.0_lon23.0to30.0.icechunk",
    "UKESM": "srm/output/qa/UKESM-ERA5-lat-31.0to-26.0_lon23.0to30.0.icechunk",
}
GCMS = list(PREFIXES)
SPATIAL = ["lat", "lon"]

## Compute

Both the scratch bucket and the cluster live in `us-west-2`. The regional box is small (roughly
20 x 28 cells vs a 721 x 1440 global grid) and there are three GCM stores instead of one, so the
cluster below is sized down from production's.

In [ ]:
# Regional run: much smaller domain x 3 GCMs, still far less data than one global GCM read.
cfg = ClusterConfig(n_workers=[2, 10])
cluster = coiled.Cluster(
    name="srm-qaqc-output-checks-regional",
    region=cfg.region,
    n_workers=cfg.n_workers,
    worker_vm_types=["r8gn.xlarge", "r8gn.2xlarge"],
    scheduler_vm_types=["c8g.large"],
    spot_policy="spot_with_fallback",
    use_best_zone=True,
    tags=cfg.tags,
)
client = hijack(cluster.get_client())
client

## The output store layout

Each GCM has its own icechunk store, versioned by branch rather than by directory path, but all
three share the same internal layout as the production store: `historical`, `ssp245`, and `g6_1p5k`
top-level groups hold the downscaled fine-grid outputs, and `debiased_coarse/…` holds the
intermediate bias-corrected coarse outputs on the GCM grid. We treat each
`(gcm, family, scenario, variable, member)` path as a single leaf and check leaves independently,
same as production.

In [ ]:
# Iterates GCMS (not PREFIXES) so trimming GCMS -- e.g. GCMS = ["CESM2-WACCM"] -- scales the
# run down to a subset of GCMs without opening stores we don't need.
trees: dict[str, xr.DataTree] = {
    gcm: _open_output_datatree(f"s3://{BUCKET}/{PREFIXES[gcm]}", branch=BRANCH) for gcm in GCMS
}
trees[GCMS[0]]

In [ ]:
SCENARIO_LABELS = {"historical": "historical", "ssp245": "SSP245", "g6_1p5k": "G6-1.5K"}
GROUP_TO_SCENARIO = {
    **SCENARIO_LABELS,
    **{f"debiased_coarse/{g}": label for g, label in SCENARIO_LABELS.items()},
}


def group_leaves(gcm: str, group: str) -> dict[tuple[str, str, str, str], xr.DataArray]:
    """One lazy DataArray per (gcm, group, variable, member).

    Variables are discovered from the tree rather than a fixed list, since the regional run's
    variable set isn't assumed to match production's.
    """
    node = trees[gcm][group]
    return {
        (gcm, group, v, m): node[f"{v}/{m}"].dataset[v]
        for v in node.children
        for m in node[v].children
    }


# One lazy DataArray per (gcm, scenario, variable, member) leaf, across both families and all GCMs.
leaves: dict[tuple[str, str, str, str], xr.DataArray] = {}
for gcm in GCMS:
    for group in GROUP_TO_SCENARIO:
        leaves.update(group_leaves(gcm, group))


def table(rows: dict) -> pd.DataFrame:
    """One row per leaf, indexed by (gcm, scenario, variable, member).

    Builds the MultiIndex explicitly rather than via ``rename_axis`` on ``from_dict``'s inferred
    index: pandas only infers a MultiIndex from tuple keys when there's at least one row.
    """
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.index = pd.MultiIndex.from_tuples(
        rows.keys(), names=["gcm", "scenario", "variable", "member"]
    )
    return df


print(f"{len(leaves)} (gcm, scenario, variable, member) leaves to check")

### A note on member coverage

This quirk is specific to `CESM2-WACCM`: `tas` uses ESM member labels (`r1i1p1f1`, …) historically
and numeric ids (`003`, `008`, …) for scenarios, while `tasmax`/`tasmin` are drawn from a corrected
model run stored as member `001` historically, because of a known CMIP6 bug (see `srm.lineage`).
`pr` and `rsds` follow `tas`'s convention. This is why CESM2-WACCM's historical period can't form a
Part 2 triplet. `MIROC-ES2H` and `UKESM` don't split members this way -- per `srm.lineage`, all
three temperature variables share the same historical member id for those two GCMs, so we'd expect
their historical periods to actually show up as checkable triplets in Part 2, unlike CESM2-WACCM's.

## Part 1: output integrity (issue #450)

Same five checks as production -- NaNs, duplicate time slices, all-zero global timesteps,
reasonable physical ranges, and time coverage against the raw input bounds -- computed once per leaf
and batched by `(gcm, scenario group)` so no single dask graph spans a whole store.

In [ ]:
def leaf_stats(da: xr.DataArray) -> xr.Dataset:
    """Lazy per-leaf reductions, fused into one pass over each array on compute."""
    return xr.Dataset(
        {
            "n_nan_cells": da.isnull().sum(),
            "min": da.min(),
            "max": da.max(),
            # Per-day spatial moments: a fingerprint reused by Checks 2, 3 and 5. Four independent
            # moments let Check 2 identify duplicate days from the fingerprint alone, with no
            # second read of the full array.
            "fingerprint": xr.concat(
                [da.min(SPATIAL), da.max(SPATIAL), da.mean(SPATIAL), da.std(SPATIAL)],
                dim=pd.Index(["min", "max", "mean", "std"], name="stat"),
            ),
        }
    )

### Check 1: no NaNs

These outputs should contain none, so we require every leaf to be entirely free of `NaN`s. A
non-zero count fails the check and points to a gap the pipeline should have filled rather than
shipped.

On a spatial subset, expect this to fail on every fine-grid leaf: downscaling leaves an always-`NaN`
border where the interpolation buffer runs off the edge of the box. `debiased_coarse` (coarse GCM
grid) isn't affected.

In [ ]:
def check_no_nans(stats: dict) -> pd.DataFrame:
    """Total NaN count per leaf; pass = zero."""
    nan_df = table({k: {"n_nan_cells": int(s.n_nan_cells)} for k, s in stats.items()})
    nan_df["pass"] = nan_df.n_nan_cells == 0
    return nan_df

### Check 2: no duplicate time slices

Two identical days usually indicate a write error, such as a chunk written with the wrong date or a
step that ran twice. Comparing every pair of days directly would scale quadratically with the length
of the record, so we reduce each day to a compact fingerprint (its spatial minimum, maximum, mean,
and standard deviation) and group the days that share one. Four independent spatial moments make an
accidental collision between two genuinely different daily fields effectively impossible, so a
shared fingerprint is reported as a duplicate directly; we deliberately avoid a second,
per-collision read of the full array, which is both an extra pass over the data and a fragile ad-hoc
compute under the distributed scheduler.

`pr` fails this on every leaf here -- Dry days in this tiny box/subset all share the same zero fingerprint, so Check2 mistakes them for duplicate writes.

In [ ]:
def duplicate_pairs(fp: xr.DataArray) -> list[tuple[str, str]]:
    """Day-pairs sharing an identical (min, max, mean, std) spatial fingerprint.

    Four independent spatial moments make an accidental collision between two genuinely different
    daily fields effectively impossible, so a shared fingerprint is reported as a duplicate directly.
    We avoid a second, per-collision ``.equals()`` read of the full array: it is both an extra pass
    over the data and a fragile ad-hoc compute under the distributed scheduler.
    """
    arr = fp.transpose("time", "stat").values
    valid = ~np.isnan(arr).any(axis=1)
    times = fp.time.values[valid]

    _, inv = np.unique(arr[valid], axis=0, return_inverse=True)
    groups: dict[int, list[np.datetime64]] = {}
    for t, g in zip(times, inv):
        groups.setdefault(int(g), []).append(t)

    def fmt(t: np.datetime64) -> str:
        return str(np.datetime_as_string(t, unit="D"))

    return [(fmt(g[0]), fmt(t)) for g in groups.values() if len(g) > 1 for t in g[1:]]


def check_no_duplicates(stats: dict) -> pd.DataFrame:
    dup_df = table(
        {k: {"duplicate_pairs": duplicate_pairs(s.fingerprint)} for k, s in stats.items()}
    )
    dup_df["pass"] = dup_df.duplicate_pairs.str.len() == 0
    return dup_df

### Check 3: reasonable ranges

We apply the same wide sanity bounds used for the input data in
[#316](https://github.com/carbonplan/srm-downscaling/issues/316). Temperatures should fall within a
broad envelope in Kelvin, and a value well outside it is more often a unit error than genuine weather. We test each leaf's global minimum and maximum against
`VAR_SPATIAL_RANGES` from `srm.qaqc`, and we separately flag any day whose spatial extreme exceeds
the outlandish thresholds of 65 °C or −100 °C. We report the flagged days for inspection rather than
failing the leaf on them, because a single implausible cell is worth seeing but does not by itself
invalidate an array.

These bounds are global, so they say nothing about a value that is globally ordinary yet locally
impossible. A 25 °C day in Antarctic winter passes here. The
[plausible-value check](plausible-value-check.ipynb) notebook is the location- and season-aware
complement to this check: it builds a per-cell, per-day-of-year envelope from the observed ERA5
record plus the coarse model's modeled change, and reports how far each leaf strays outside it.

`VAR_SPATIAL_RANGES` now also carries an entry for `rsds`: its lower bound allows the leaf's global
minimum down to −1 W m⁻² rather than requiring strict non-negativity, since downscaled shortwave
radiation was slightly negative in ERA5. This is the same kind of near-zero
excursion `pr`'s Part 3 check tracks explicitly on a per-cell, per-day basis -- for `rsds`, only
this coarse global-extreme test currently watches for it.

In [ ]:
TEMP_VARS = {"tas", "tasmax", "tasmin"}
HOT_K = 65 + 273.15  # outlandishly_high_temp threshold (srm.qaqc)
COLD_K = -100 + 273.15  # outlandishly_low_temp threshold (srm.qaqc)


def irregular_days(v: str, fp: xr.DataArray) -> list[tuple[str, float]]:
    """(date, K) for days whose spatial max/min breaches the outlandish temp thresholds."""
    if v not in TEMP_VARS:
        return []
    smax = fp.sel(stat="max")
    smin = fp.sel(stat="min")
    bad = ((smax > HOT_K) | (smin < COLD_K)).values
    times = fp.time.values[bad]
    vals = smax.where(smax > HOT_K, smin).values[bad]  # report whichever extreme tripped
    return [(str(t)[:10], round(float(x), 1)) for t, x in zip(times, vals)]


def check_reasonable_ranges(stats: dict) -> pd.DataFrame:
    range_df = table(
        {
            k: {
                "min": float(s["min"]),
                "max": float(s["max"]),
                "irregular_days": irregular_days(k[2], s["fingerprint"]),
            }
            for k, s in stats.items()
        }
    )
    range_df["pass"] = [
        VAR_SPATIAL_RANGES[v]["min"][0] <= mn <= VAR_SPATIAL_RANGES[v]["min"][1]
        and VAR_SPATIAL_RANGES[v]["max"][0] <= mx <= VAR_SPATIAL_RANGES[v]["max"][1]
        for (gcm, s, v, m), mn, mx in zip(range_df.index, range_df["min"], range_df["max"])
    ]
    range_df["n_irregular_days"] = range_df.irregular_days.str.len()
    return range_df

### Check 4: within input time bounds

Downscaled output should not extend beyond the input it was derived from. We compare each leaf's
first/last timestamp against `resolve_member_time_bounds(gcm, scenario, member)` -- already
GCM-aware -- and fail any leaf that starts before or ends after those bounds, with no exemptions.
`bridged_pre_sai` and `stale_end_tail` are the same regression-guard sentinels production uses.

In [ ]:
def check_time_bounds(leaves: dict) -> pd.DataFrame:
    bounds_df = table(
        {
            (gcm, s, v, m): {
                "actual_start": str(da.time.values[0])[:10],
                "actual_end": str(da.time.values[-1])[:10],
                "n_time": da.sizes["time"],
                "expected": resolve_member_time_bounds(gcm, GROUP_TO_SCENARIO[s], m),
            }
            for (gcm, s, v, m), da in leaves.items()
        }
    )
    expected_start = bounds_df.expected.str[0].fillna("0000-01-01")  # no known bounds -> pass
    expected_end = bounds_df.expected.str[1].fillna("9999-12-31")
    is_sai = bounds_df.index.get_level_values("scenario").str.upper().str.contains("G6|SAI")
    starts_before_raw = bounds_df.actual_start < expected_start
    ends_after_raw = bounds_df.actual_end > expected_end
    # Same sentinels as production: bridged_pre_sai flags a SAI leaf starting before its raw input
    # bound (pre-SAI bridge); stale_end_tail flags a leaf running past its bound. No exemptions.
    bounds_df["bridged_pre_sai"] = is_sai & starts_before_raw
    bounds_df["stale_end_tail"] = ends_after_raw
    bounds_df["pass"] = ~starts_before_raw & ~ends_after_raw
    return bounds_df

### Check 5: no all-zero timesteps

No physical field should ever be identically zero across the entire globe on a given day: a day of
0 K everywhere, or of zero precipitation, solar radiation, or humidity at every cell, is not weather
but a write or masking error (issue #425). We reuse the spatial minimum and maximum from the per-day
fingerprint of Check 2, and flag any day whose minimum and maximum are both exactly zero, which can
only happen when every cell is zero. The check applies to all variables, including precipitation:
although `pr` is legitimately zero at many individual cells, an entire global field of zero on a
single day never occurs in a correct full-globe dataset.

That last assumption doesn't hold on a small, semi-arid box: whole-domain-zero-rain days on `pr` are
physically real here, not a write error. Check 2's duplicate-day flags on `pr` are the same
phenomenon.

In [ ]:
def all_zero_days(fp: xr.DataArray) -> list[str]:
    """Dates whose entire spatial field is exactly 0 (spatial min and max both 0) -- see #425."""
    smin = fp.sel(stat="min")
    smax = fp.sel(stat="max")
    bad = ((smin == 0) & (smax == 0)).values
    return [str(t)[:10] for t in fp.time.values[bad]]


def check_no_all_zero(stats: dict) -> pd.DataFrame:
    """Per-leaf list of all-zero global days; pass = none."""
    zero_df = table(
        {k: {"all_zero_days": all_zero_days(s["fingerprint"])} for k, s in stats.items()}
    )
    zero_df["pass"] = zero_df.all_zero_days.str.len() == 0
    return zero_df

### Running the checks

Same batched approach as production, with an added outer loop over `GCMS`: materialize per-leaf
stats one `(gcm, scenario group)` at a time, across several small `dask.compute` calls, then run the
five checks plus Part 3's precip physical counts against the collected summaries.

In [ ]:
%%time

# Per-leaf reductions, batched by (gcm, scenario group) so no single dask graph spans every leaf.
# We also fold precipitation's Part-3 physical counts (negative / outlandish days per cell) into
# each group's read, so every pr array is read exactly once here rather than again in Part 3.
PR_OUTLANDISH = 2000 / 86400  # 2000 mm/day in kg m-2 s-1 (~0.02315); largest recorded ~1825 mm/day


def pr_physical(da: xr.DataArray) -> xr.Dataset:
    """Per-cell day counts for the Part-3 precip checks, folded into the Part-1 read."""
    return xr.Dataset(
        {
            "neg_days": (da < 0).sum("time"),  # non-negativity: expected 0 everywhere
            "high_days": (da > PR_OUTLANDISH).sum("time"),  # outlandish daily total
        },
        attrs={"n_time": da.sizes["time"]},
    )


def compute_with_retry(*collections, attempts=3, label=""):
    """dask.compute with retries so a transient scheduler glitch does not abort the whole run."""
    import time

    for attempt in range(1, attempts + 1):
        try:
            return dask.compute(*collections)
        except Exception as exc:
            if attempt == attempts:
                raise
            print(f"    {label}retry {attempt}/{attempts - 1} after {type(exc).__name__}")
            time.sleep(5)


# Compute each (gcm, scenario group) independently: a transient failure retries, and a group that
# still fails is recorded and skipped so the rest of the notebook still runs on what did compute.
stats: dict = {}
pr_phys: dict = {}
failed_groups: list[tuple[str, str]] = []
for gcm in GCMS:
    for group in GROUP_TO_SCENARIO:
        g_leaves = {k: da for k, da in leaves.items() if k[0] == gcm and k[1] == group}
        if not g_leaves:
            continue
        try:
            s, p = compute_with_retry(
                {k: leaf_stats(da) for k, da in g_leaves.items()},
                {k: pr_physical(da) for k, da in g_leaves.items() if k[2] == "pr"},
                label=f"[{gcm}/{group}] ",
            )
        except Exception as exc:
            failed_groups.append((gcm, group))
            print(
                f"  {gcm}/{group}: FAILED after retries ({type(exc).__name__}); continuing with the rest"
            )
            continue
        stats.update(s)
        pr_phys.update(p)
        print(f"  {gcm}/{group}: {len(s)} leaves computed")

if failed_groups:
    print(
        f"\n!! groups that did not compute: {failed_groups} -- rerun those before trusting results"
    )

# Every check runs on the leaves that actually computed, so a failed group is simply absent (with the
# warning above) rather than crashing the notebook. bounds only reads time coords, so restrict it too.
computed = {k: leaves[k] for k in stats}
nan_df = check_no_nans(stats)
dup_df = check_no_duplicates(stats)
zero_df = check_no_all_zero(stats)
range_df = check_reasonable_ranges(stats)
bounds_df = check_time_bounds(computed)

summary = pd.concat(
    {
        "no_nans": nan_df["pass"],
        "no_duplicate_timesteps": dup_df["pass"],
        "no_all_zero_days": zero_df["pass"],
        "reasonable_range": range_df["pass"],
        "n_irregular_days": range_df["n_irregular_days"],
        "within_input_time_bounds": bounds_df["pass"],
        "bridged_pre_sai": bounds_df["bridged_pre_sai"],
        "stale_end_tail": bounds_df["stale_end_tail"],
    },
    axis=1,
)
print(summary.to_string())

check_cols = [
    "no_nans",
    "no_duplicate_timesteps",
    "no_all_zero_days",
    "reasonable_range",
    "within_input_time_bounds",
]
print(f"\nAll checks pass across {len(summary)} leaves: {bool(summary[check_cols].all().all())}")

tb_fail = ~summary["within_input_time_bounds"]
known = summary["bridged_pre_sai"] | summary["stale_end_tail"]
print(
    f"within_input_time_bounds failures: {int(tb_fail.sum())} "
    f"(g6 bridge: {int(summary['bridged_pre_sai'].sum())}, "
    f"stale end tail: {int(summary['stale_end_tail'].sum())}; "
    f"all known: {bool((tb_fail == known).all())})"
)
other_cols = ["no_nans", "no_duplicate_timesteps", "no_all_zero_days", "reasonable_range"]
print(f"All other checks pass on every leaf: {bool(summary[other_cols].all().all())}")

A clean store reports zero `NaN`s, no duplicated days, no all-zero global fields, and values within
range. On this box, expect Check 1 to fail on every fine-grid leaf and Checks 2/5 to fail on every
`pr` leaf for the reasons noted above -- Checks 3 and 4 should still pass cleanly. The drill-down
below surfaces any all-zero timesteps (Check 5, issue #425) and lists any individual days whose
spatial extreme crossed the outlandish thresholds, shown for inspection rather than as failures.

In [ ]:
# Any leaf with NaNs fails Check 1.
print(f"Leaves with any NaN: {int((nan_df.n_nan_cells > 0).sum())}")

# Any leaf with an all-zero global timestep fails Check 5 (#425).
zero_flagged = zero_df[~zero_df["pass"]]
if len(zero_flagged):
    print("\nAll-zero timesteps (Check 5 failure, #425):")
    for (gcm, s, v, m), row in zero_flagged.iterrows():
        print(f"  {gcm}/{s}/{v}/{m}: {len(row.all_zero_days)} days -- e.g. {row.all_zero_days[:5]}")
else:
    print("\nNo all-zero timesteps flagged.")

# Drill-down: dates + values for any leaf with implausible single-day spikes.
flagged = range_df[range_df.n_irregular_days > 0]
if len(flagged):
    print("\nIrregular days (report-only, spatial extreme past 65C / -100C):")
    for (gcm, s, v, m), row in flagged.iterrows():
        print(f"  {gcm}/{s}/{v}/{m}: {row.n_irregular_days} days")
        for date, k in row.irregular_days:
            print(f"    {date}  {k}K ({k - 273.15:.1f}C)")
else:
    print("\nNo irregular days flagged.")

### Time coverage per leaf

Lists each leaf's real first/last day, `n_time`, the expected bounds from
`resolve_member_time_bounds`, and the two out-of-bounds sentinels -- per GCM, since members and their
expected bounds differ by GCM (numeric ids for CESM2-WACCM, `r0X` for MIROC-ES2H, ESM labels for
UKESM).

In [ ]:
# Per-leaf time coverage, per GCM: each leaf keeps its own real extent (no shared, padded axis).
coverage_cols = [
    "actual_start",
    "actual_end",
    "n_time",
    "expected",
    "bridged_pre_sai",
    "stale_end_tail",
]
print(bounds_df[coverage_cols].to_string())

## Part 2: temperature ordering consistency (issue #448)

Same check as production: `tasmin ≤ tas ≤ tasmax` should hold. `tasmax ≥ tasmin` is enforced by
post-processing ([#331](https://github.com/carbonplan/srm-downscaling/issues/331)) and should hold
everywhere; `tasmax ≥ tas` and `tasmin ≤ tas` are unguaranteed and only measured, per GCM.

### Assembling comparable triplets

We intersect the members that carry `tas`, `tasmax`, and `tasmin` together within each
`(gcm, group)`, same as production. This excludes CESM2-WACCM's historical period, per the
member-coverage note above.

In [ ]:
%%time


def temperature_triplets(leaves: dict) -> list[tuple[str, str, str]]:
    """(gcm, family/scenario group, member) paths that carry tas AND tasmax AND tasmin."""
    have: dict[tuple[str, str, str], set[str]] = {}
    for gcm, group, v, m in leaves:
        have.setdefault((gcm, group, m), set()).add(v)
    return sorted(gm for gm, vs in have.items() if TEMP_VARS <= vs)


def ordering_violations(gcm: str, group: str, member: str) -> xr.Dataset:
    """Per-cell count of days each temperature-ordering rule is broken, for one triplet.

    tas/tasmax/tasmin can cover different time spans within one member (e.g. tas may end a
    year earlier), so we inner-join on time before comparing.
    """
    node = trees[gcm][group]
    tas = node[f"tas/{member}"].dataset["tas"]
    tasmax = node[f"tasmax/{member}"].dataset["tasmax"]
    tasmin = node[f"tasmin/{member}"].dataset["tasmin"]
    tas, tasmax, tasmin = xr.align(tas, tasmax, tasmin, join="inner")
    return xr.Dataset(
        {
            "max_lt_min": (tasmax < tasmin).sum("time"),  # expected 0 (#331 post-processing)
            "max_lt_tas": (tasmax < tas).sum("time"),  # diagnostic (#448)
            "min_gt_tas": (tasmin > tas).sum("time"),  # diagnostic (#448)
        },
        attrs={"n_time": tas.sizes["time"]},
    )


triplets = temperature_triplets(leaves)
print("Checkable (gcm, group, member) triplets:")
for gcm, g, m in triplets:
    print(f"  {gcm}/{g}/{m}")

viol = compute_with_retry({gm: ordering_violations(*gm) for gm in triplets}, label="[part2] ")[0]

### Violation frequencies

Reports the number of days on which `tasmax < tasmin` (expected zero), and for the two unguaranteed
relationships the number of grid cells violated on at least one day, with the fraction of the domain
they represent -- per `(gcm, group, member)` triplet. That fraction saturates toward one over a long
record (a cell counts as soon as it fails once), so the maps and area-weighted table that follow are
the fairer read of severity.

In [ ]:
def consistency_summary(viol: dict) -> pd.DataFrame:
    rows = {}
    for (gcm, group, m), ds in viol.items():
        rows[(gcm, group, m)] = {
            "n_time": ds.attrs["n_time"],
            "days_max<min": int(ds.max_lt_min.sum()),  # #331 guarantee: should be 0
            "cells_max<tas": int((ds.max_lt_tas > 0).sum()),
            "frac_cells_max<tas": round(float((ds.max_lt_tas > 0).mean()), 5),
            "cells_min>tas": int((ds.min_gt_tas > 0).sum()),
            "frac_cells_min>tas": round(float((ds.min_gt_tas > 0).mean()), 5),
        }
    # Build the MultiIndex explicitly (see `table`'s docstring).
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.index = pd.MultiIndex.from_tuples(rows.keys(), names=["gcm", "group", "member"])
    return df


consistency_df = consistency_summary(viol)
print(consistency_df.to_string())

n_regressions = int(consistency_df["days_max<min"].sum())
print(
    f"\ntasmax >= tasmin holds everywhere (#331): {n_regressions == 0} "
    f"(total offending days across triplets: {n_regressions})"
)

### Where the ordering breaks

Maps the fraction of days `tasmax < tas` and `tasmin > tas`, per `(gcm, group, member)` triplet,
masking cells that never violate the ordering. A pale field means the ordering almost always holds;
a saturated one means it fails often. This box is small (interior South Africa) and away from the
coastal-upwelling and high-latitude bands where production's global maps show the largest effect, so
expect much sparser violations here.

In [ ]:
def plot_violation_maps(viol: dict) -> None:
    conds = [("max_lt_tas", "days tasmax < tas"), ("min_gt_tas", "days tasmin > tas")]
    for (gcm, group, m), ds in viol.items():
        n_time = ds.attrs["n_time"]
        fig, axes = plt.subplots(
            1, 2, figsize=(16, 4), subplot_kw={"projection": ccrs.PlateCarree()}
        )
        for ax, (cond, title) in zip(axes, conds):
            frac = (ds[cond] / n_time).where(ds[cond] > 0)
            frac.plot(
                ax=ax,
                transform=ccrs.PlateCarree(),
                cmap="magma_r",
                cbar_kwargs={"label": "fraction of days", "shrink": 0.8},
            )
            ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.4")
            ax.gridlines(draw_labels=False, color="0.9", linewidth=0.4)
            ax.set_title(f"{gcm}/{group}/{m}\n{title}")
        plt.tight_layout()
        plt.show()


plot_violation_maps(viol)

### Land, ocean, and high latitudes

Same stratification as production, unchanged: for each `(gcm, group, member)` triplet we split the
two diagnostic conditions across five regions (global, land, ocean, land/ocean below 60° latitude,
poleward of 60°), area-weighted by cos(latitude), using a Natural Earth land mask on cell centres.
This regional box has no polar cells and a small, possibly all-land or all-ocean footprint, so expect
the `|lat|>=60` row and the land/ocean split to be sparse or degenerate here -- that's a fact about
this box, not a reason to change the check, which should behave normally again on a full-globe run.

In [ ]:
# Land/ocean and high-latitude stratification of the Part-2 counts. This is pure post-processing of
# the per-cell day counts already in `viol`, so it triggers no further reads of the store. Unchanged
# from production -- see the markdown above for why results are expected to be sparse on this box.
_LAND_MASKS: dict[tuple[int, int], xr.DataArray] = {}


@functools.cache
def _land_polygons():
    """Union of the Natural Earth 50 m land polygons, cached for the session."""
    path = shpreader.natural_earth(resolution="50m", category="physical", name="land")
    return shapely.union_all(list(shpreader.Reader(path).geometries()))


def land_mask(template: xr.DataArray) -> xr.DataArray:
    """Boolean mask on template's grid, True where the cell centre falls on land.

    Unchanged from production: test cell centres against Natural Earth rather than the pipeline's
    optional ocean mask, which carries no Antarctica and rasterizes with all_touched=True. Cached per
    grid shape, since the coarse/fine families (and different GCM grids) differ.
    """
    key = (template.sizes["lat"], template.sizes["lon"])
    if key not in _LAND_MASKS:
        lon, lat = np.meshgrid(template["lon"].values, template["lat"].values)
        _LAND_MASKS[key] = xr.DataArray(
            shapely.contains_xy(_land_polygons(), lon, lat),
            coords={"lat": template["lat"], "lon": template["lon"]},
            dims=("lat", "lon"),
        )
    return _LAND_MASKS[key]


def regions_for(land: xr.DataArray) -> dict[str, xr.DataArray]:
    """Boolean region selectors on the land mask's grid, in reporting order."""
    abslat = np.abs(land["lat"])
    everywhere = xr.ones_like(land)
    return {
        "global": everywhere,
        "land": land,
        "ocean": ~land,
        "land |lat|<60": land & (abslat < 60),
        "ocean |lat|<60": ~land & (abslat < 60),
        "|lat|>=60": everywhere & (abslat >= 60),
    }


CONDS = {"max<tas": "max_lt_tas", "min>tas": "min_gt_tas"}


def regional_consistency_summary(viol: dict) -> pd.DataFrame:
    """Violation statistics per triplet and region, area-weighted by cos(lat). Unchanged from
    production -- see the markdown above for why regions are expected to be sparse here."""
    rows = {}
    for (gcm, group, member), ds in viol.items():
        counts = ds["max_lt_tas"]
        land = land_mask(counts)
        weights = np.cos(np.deg2rad(ds["lat"])).broadcast_like(counts)
        n_time = ds.attrs["n_time"]
        for name, sel in regions_for(land).items():
            w = weights.where(sel, 0.0)
            row = {"area_frac": round(float(w.sum() / weights.sum()), 4)}
            for label, cond in CONDS.items():
                ever = (ds[cond] > 0).astype("float32")
                freq = ds[cond] / n_time
                row[f"frac_area_ever_{label}"] = round(float(ever.weighted(w).mean(SPATIAL)), 5)
                row[f"mean_freq_{label}"] = round(float(freq.weighted(w).mean(SPATIAL)), 6)
            row["days_max<min"] = int(ds["max_lt_min"].where(sel, 0).sum())
            rows[(gcm, group, member, name)] = row
    # Build the MultiIndex explicitly (see `table`'s docstring).
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.index = pd.MultiIndex.from_tuples(rows.keys(), names=["gcm", "group", "member", "region"])
    return df


regional_df = regional_consistency_summary(viol)
print(regional_df.to_string())

# The headline for the land-vs-ocean reading: how many times more often a unit of ocean area breaks
# the ordering than a unit of land area does. "land 0" means the ordering never broke over land.
print("\nocean:land ratio of mean violation frequency")
for gcm, group, member in viol:
    parts = []
    for label in CONDS:
        over_land = regional_df.loc[(gcm, group, member, "land"), f"mean_freq_{label}"]
        over_ocean = regional_df.loc[(gcm, group, member, "ocean"), f"mean_freq_{label}"]
        parts.append(f"{label}: {over_ocean / over_land:.1f}x" if over_land else f"{label}: land 0")
    print(f"  {gcm}/{group}/{member}  " + "   ".join(parts))

Interpretation depends on this run's actual table above -- production's specific ratios (7-8x
ocean:land for `tasmin > tas`, poleward concentration for `tasmax < tas`) came from a full-globe read
and won't transfer to this small box. Read the `mean_freq_*` and `frac_area_ever_*` columns per
`gcm`/`group`/`member` directly once this has run.

## Part 3: precipitation physical constraints (issue #459)

Same single-variable checks as production -- `pr` must never go negative, and no day may carry an
outlandishly large total -- counted per grid cell in the same Part-1 pass, per GCM. This section only
summarizes and maps the results.

In [ ]:
def pr_physical_summary(phys: dict) -> pd.DataFrame:
    """One row per pr leaf: negative and outlandish-high day/cell tallies."""
    rows = {}
    for (gcm, group, _v, m), ds in phys.items():
        rows[(gcm, group, m)] = {
            "n_time": ds.attrs["n_time"],
            "neg_cell_days": int(ds.neg_days.sum()),
            "cells_with_neg": int((ds.neg_days > 0).sum()),
            "frac_cells_neg": round(float((ds.neg_days > 0).mean()), 5),
            "high_cell_days": int(ds.high_days.sum()),
            "cells_with_high": int((ds.high_days > 0).sum()),
        }
    # Build the MultiIndex explicitly (see `table`'s docstring).
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.index = pd.MultiIndex.from_tuples(rows.keys(), names=["gcm", "group", "member"])
    return df


pr_physical_df = pr_physical_summary(pr_phys)
print(pr_physical_df.to_string())

n_neg = int(pr_physical_df["neg_cell_days"].sum())
n_high = int(pr_physical_df["high_cell_days"].sum())
print(f"\nNon-negativity holds everywhere (pr >= 0): {n_neg == 0} (negative cell-days: {n_neg})")
print(f"No outlandish daily totals: {n_high == 0} (outlandish cell-days: {n_high})")

In [ ]:
def plot_negativity_maps(phys: dict) -> None:
    """Map the fraction of days pr < 0, for any leaf that ever goes negative."""
    plotted = False
    for (gcm, group, _v, m), ds in phys.items():
        if int((ds.neg_days > 0).sum()) == 0:
            continue
        plotted = True
        frac = (ds.neg_days / ds.attrs["n_time"]).where(ds.neg_days > 0)
        fig, ax = plt.subplots(figsize=(8, 4), subplot_kw={"projection": ccrs.PlateCarree()})
        frac.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap="magma_r",
            cbar_kwargs={"label": "fraction of days pr < 0", "shrink": 0.8},
        )
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="0.4")
        ax.gridlines(draw_labels=False, color="0.9", linewidth=0.4)
        ax.set_title(f"{gcm}/{group}/{m}\nnegative precipitation")
        plt.tight_layout()
        plt.show()
    if not plotted:
        print("No leaf has any negative precipitation — nothing to map.")


def pr_high_days(fp: xr.DataArray) -> list[tuple[str, float]]:
    """(date, kg m-2 s-1) for days whose spatial-max pr exceeds the outlandish threshold."""
    smax = fp.sel(stat="max")
    bad = (smax > PR_OUTLANDISH).values
    return [(str(t)[:10], float(x)) for t, x in zip(fp.time.values[bad], smax.values[bad])]


plot_negativity_maps(pr_phys)

# Report-only drill-down over the Part-1 fingerprints, restricted to pr leaves.
flagged = {k: pr_high_days(s.fingerprint) for k, s in stats.items() if k[2] == "pr"}
flagged = {k: days for k, days in flagged.items() if days}
if flagged:
    print("Outlandish precip days (report-only, spatial max > 2000 mm/day):")
    for (gcm, s, v, m), days in flagged.items():
        print(f"  {gcm}/{s}/{v}/{m}: {len(days)} days")
        for date, val in days:
            print(f"    {date}  {val:.3e} kg m-2 s-1 ({val * 86400:.0f} mm/day)")
else:
    print("No outlandish precip days flagged.")

## Summary

Same three-part structure as production, run per GCM against the regional (wide) run instead of one
full-globe store:

- **Part 1** (issue #450): same five checks as production. Expect Check 1 to fail on every
  fine-grid leaf and Checks 2/5 to fail on every `pr` leaf for this box, per the caveats above;
  Checks 3 and 4 should pass cleanly.
- **Part 2** (issue #448): `tasmin ≤ tas ≤ tasmax` ordering, per `(gcm, group, member)` triplet.
  `tasmax ≥ tasmin` should hold everywhere (#331); the other two relationships are diagnostic only.
  MIROC-ES2H and UKESM should surface historical triplets that CESM2-WACCM cannot (see the
  member-coverage note above). Land/ocean stratification is expected to be degenerate on this
  land-locked box.
- **Part 3** (issue #459): precipitation non-negativity and outlandish-total checks, per `pr` leaf.

This notebook doesn't carry forward production's specific findings (leaf counts, violation ratios,
regression history) since those describe a different store at a different scale -- see the tables
and prints above for this run's actual numbers.

In [ ]:
cluster.shutdown()